# XGBoost development and feature exploration

Compare the saved model configurations and inspect validation, demand-group, and test errors. The initial count objective and later squared-error objective are different configurations; their comparison is not a pure feature ablation. Clipping is applied only where explicitly shown.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.compose import ColumnTransformer


In [ ]:
df = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_features.parquet"))


In [ ]:
X = df.copy()

X_train = X[X["tpep_pickup_datetime"].dt.month <= 8]
X_val = X[(X["tpep_pickup_datetime"].dt.month > 8)&(X["tpep_pickup_datetime"].dt.month <= 9)]
X_test = X[X["tpep_pickup_datetime"].dt.month > 9]
y_train = X_train.trip_count
y_val = X_val.trip_count
y_test = X_test.trip_count


In [ ]:
train = X_train.drop(columns = ["trip_count"])
val = X_val.drop(columns = ["trip_count"])
test = X_test.drop(columns = ["trip_count"])

train = train.drop(columns = ["tpep_pickup_datetime"])
val = val.drop(columns = ["tpep_pickup_datetime"])
test = test.drop(columns = ["tpep_pickup_datetime"])


In [ ]:
categorical = ["PULocationID", "hours"]
numerical = [
    col for col in train.columns
    if col not in categorical
]

preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown = "ignore"), categorical),
    ("numbers", StandardScaler(), numerical),
])
model = make_pipeline(
    preprocessor,
    xgb.XGBRegressor(
        objective = "count:poisson",
        random_state = 42,

    )
)

model.fit(train, y_train)


In [ ]:
y_pred = model.predict(val)


In [ ]:
mae = mean_absolute_error(y_val, y_pred)
rsme = root_mean_squared_error(y_val, y_pred)
m_pred = y_pred.mean()
med_pred = np.median(y_pred)
max_pred = np.max(y_pred)
p95_pred = np.percentile(y_pred, 95)
p99_pred = np.percentile(y_pred, 99)

print("Validation performance")
print(f"MAE:  {mean_absolute_error(y_val, y_pred):.4f}")
print(f"RMSE: {root_mean_squared_error(y_val, y_pred):.4f}")
print("\nPredicted pickups per zone per hour")
print(f"Mean:            {m_pred:.4f}")
print(f"Median:          {med_pred:.4f}")
print(f"95th percentile: {p95_pred:.4f}")
print(f"99th percentile: {p99_pred:.4f}")
print(f"Maximum:         {max_pred:.4f}")


## Comparison with the Poisson regression baseline


In [ ]:
# Match predictions to validation rows before aggregating.
pickup_comparison = X_val[["tpep_pickup_datetime", "PULocationID"]].copy()
pickup_comparison["Actual"] = y_val.loc[pickup_comparison.index]
pickup_comparison["Predicted"] = np.asarray(y_pred)

hourly_comparison = pickup_comparison.groupby("tpep_pickup_datetime")[["Actual", "Predicted"]].sum().sort_index()
zone_comparison = pickup_comparison.groupby("PULocationID")[["Actual", "Predicted"]].sum().sort_index()

fig, ax = plt.subplots(figsize = (14, 4))
ax.plot(hourly_comparison.index, hourly_comparison["Actual"], label = "Actual", color = "tab:blue")
ax.plot(hourly_comparison.index, hourly_comparison["Predicted"], label = "Predicted", color = "tab:orange")
ax.set_title("Total pickups across all zones at each validation hour")
ax.set_xlabel("Pickup date and hour")
ax.set_ylabel("Total pickups")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize = (14, 4))
zone_positions = np.arange(len(zone_comparison))
ax.bar(zone_positions - 0.2, zone_comparison["Actual"], width = 0.4, label = "Actual", color = "tab:blue")
ax.bar(zone_positions + 0.2, zone_comparison["Predicted"], width = 0.4, label = "Predicted", color = "tab:orange")
ax.set_xticks(zone_positions[::10], zone_comparison.index[::10], rotation = 45)
ax.set_title("Total pickups per zone over the validation period")
ax.set_xlabel("Pickup zone ID")
ax.set_ylabel("Total pickups")
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
pair_important_feat = []
for x, y in zip(model.named_steps["xgbregressor"].feature_importances_, model.named_steps["columntransformer"].get_feature_names_out()):
    pair_important_feat.append((x, y))
L_sorted = sorted(pair_important_feat, key = lambda x:x[0], reverse = True)


In [ ]:
train


#### Feature ablation


In [ ]:
ablation = ["month"]
train_ablation = train.drop(columns = ablation)

categorical = ["PULocationID", "hours"]
numerical = [
    col for col in train_ablation.columns
    if (col not in categorical and col not in ablation)
]

preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown = "ignore"), categorical),
    ("numbers", StandardScaler(), numerical),
])
model = make_pipeline(
    preprocessor,
    xgb.XGBRegressor(
        objective = "reg:squarederror",
        random_state = 42
    )
)

model.fit(train_ablation, y_train)


In [ ]:
y_pred = model.predict(val).clip(min = 0)


In [ ]:
mae = mean_absolute_error(y_val, y_pred)
rsme = root_mean_squared_error(y_val, y_pred)
m_pred = y_pred.mean()
med_pred = np.median(y_pred)
max_pred = np.max(y_pred)
p95_pred = np.percentile(y_pred, 95)
p99_pred = np.percentile(y_pred, 99)

print("Validation performance")
print(f"MAE:  {mean_absolute_error(y_val, y_pred):.4f}")
print(f"RMSE: {root_mean_squared_error(y_val, y_pred):.4f}")
print("\nPredicted pickups per zone per hour")
print(f"Mean:            {m_pred:.4f}")
print(f"Median:          {med_pred:.4f}")
print(f"95th percentile: {p95_pred:.4f}")
print(f"99th percentile: {p99_pred:.4f}")
print(f"Maximum:         {max_pred:.4f}")


In [ ]:
#Ranking the highest demand on an hourly average
rank = []
for z in X_train.PULocationID.unique():
    rank.append((z, X_train["trip_count"][X_train["PULocationID"] == z].mean()))

rank = sorted(rank, key = lambda x:x[1])
rank.reverse()

rank_zones = [rank[i][0] for i in range(len(rank))] #keep only the zone number
high_demand = rank_zones[:(263//3)]
mid_demand = rank_zones[(263//3):2*(263//3)]
low_demand = rank_zones[2*(263//3):]


In [ ]:
val_high = val[val["PULocationID"].isin(high_demand)]
val_mid = val[val["PULocationID"].isin(mid_demand)]
val_low = val[val["PULocationID"].isin(low_demand)]

y_high = X_val.trip_count[X_val["PULocationID"].isin(high_demand)]
y_pred_high = model.predict(val_high)

y_mid = X_val.trip_count[X_val.PULocationID.isin(mid_demand)]
y__pred_mid = model.predict(val_mid)

y_low = X_val.trip_count[X_val.PULocationID.isin(low_demand)]
y_pred_low = model.predict(val_low)


In [ ]:
mae_high = mean_absolute_error(y_high, y_pred_high)
rmse_high = root_mean_squared_error(y_high, y_pred_high)
mae_mid = mean_absolute_error(y_mid, y__pred_mid)
rmse_mid = root_mean_squared_error(y_mid, y__pred_mid)
mae_low = mean_absolute_error(y_low, y_pred_low)
rmse_low = root_mean_squared_error(y_low, y_pred_low)

print(f"Validation results — XGBoost (removed features: {', '.join(ablation) or 'none'})")
print(f"High demand zones: MAE = {mae_high:.4f}, RMSE = {rmse_high:.4f}")
print(f"Mid demand zones:  MAE = {mae_mid:.4f}, RMSE = {rmse_mid:.4f}")
print(f"Low demand zones:  MAE = {mae_low:.4f}, RMSE = {rmse_low:.4f}")


In [ ]:
# Mean signed error: positive = overprediction; negative = underprediction.
bias_high = np.mean(np.asarray(y_pred_high) - np.asarray(y_high))
bias_mid = np.mean(np.asarray(y__pred_mid) - np.asarray(y_mid))
bias_low = np.mean(np.asarray(y_pred_low) - np.asarray(y_low))

print("Validation bias — predicted minus actual pickups per zone per hour")
print(f"High demand zones: Bias = {bias_high:+.4f}")
print(f"Mid demand zones:  Bias = {bias_mid:+.4f}")
print(f"Low demand zones:  Bias = {bias_low:+.4f}")


### Bias interpretation by demand group

1. **High-demand zones:** Average demand is approximately **56.8 pickups per zone per hour**. A bias of **−1.09** represents underprediction of only about **1.9%** of average demand.
2. **Medium-demand zones:** Bias is negligible relative to average demand.
3. **Low-demand zones:** Average demand is approximately **0.137 pickups per zone per hour**. A bias of **+0.0304** represents overprediction of about **22%** of average demand.

Bias is calculated as **predicted − actual**. These percentages compare mean bias with mean demand; they are not mean absolute percentage errors.


In [ ]:
len([x for x in y_pred if x < 0])


#### Evaluation


In [ ]:
y_pred_test = model.predict(test).clip(min = 0)


In [ ]:
mae = mean_absolute_error(y_test, y_pred_test)
rsme = root_mean_squared_error(y_test, y_pred_test)
m_pred = y_pred_test.mean()
med_pred = np.median(y_pred_test)
max_pred = np.max(y_pred_test)
p95_pred = np.percentile(y_pred_test, 95)
p99_pred = np.percentile(y_pred_test, 99)

print("Test performance — October–December 2024")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rsme:.4f}")
print("\nPredicted pickups per zone per hour")
print(f"Mean:            {m_pred:.4f}")
print(f"Median:          {med_pred:.4f}")
print(f"95th percentile: {p95_pred:.4f}")
print(f"99th percentile: {p99_pred:.4f}")
print(f"Maximum:         {max_pred:.4f}")


In [ ]:
# Evaluate the fixed model on the test period; do not refit it here.
assert test.index.equals(y_test.index)
assert X_test.index.equals(test.index)
assert len(y_pred_test) == len(y_test)
test_comparison = X_test[["tpep_pickup_datetime", "PULocationID"]].copy()
test_comparison["Actual"] = y_test.loc[test_comparison.index]
test_comparison["Predicted"] = np.asarray(y_pred_test)

def test_metric_summary(label, rows):
    actual = rows["Actual"].to_numpy()
    predicted = rows["Predicted"].to_numpy()
    errors = predicted - actual
    return {
        "Group": label,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": root_mean_squared_error(actual, predicted),
        "Bias": np.mean(errors),
        "P95 absolute error": np.percentile(np.abs(errors), 95),
        "P99 absolute error": np.percentile(np.abs(errors), 99),
    }

metric_rows = [test_metric_summary("All zones", test_comparison)]
for label, zones in [("High demand", high_demand), ("Mid demand", mid_demand), ("Low demand", low_demand)]:
    metric_rows.append(test_metric_summary(label, test_comparison.loc[test_comparison["PULocationID"].isin(zones)]))
zone_131_test = test_comparison.loc[test_comparison["PULocationID"] == 131].sort_values("tpep_pickup_datetime")
metric_rows.append(test_metric_summary("Zone 131", zone_131_test))
test_metrics = pd.DataFrame(metric_rows).set_index("Group")
print("Test results — October–December 2024")
print("Errors are in pickups per zone per hour; bias = predicted − actual.")
print(test_metrics.to_string(float_format = lambda value: f"{value:.4f}"))

week_start = zone_131_test["tpep_pickup_datetime"].min().normalize()
week_end = week_start + pd.Timedelta(days = 7)
zone_131_week = zone_131_test.loc[zone_131_test["tpep_pickup_datetime"] < week_end]
fig, axes = plt.subplots(2, 1, figsize = (14, 8))
for ax, rows, title in [
    (axes[0], zone_131_test, "Zone 131: hourly pickups over the full test period (October–December 2024)"),
    (axes[1], zone_131_week, f"Zone 131: first test week ({week_start:%d %b}–{week_end - pd.Timedelta(days = 1):%d %b %Y})"),
]:
    ax.plot(rows["tpep_pickup_datetime"], rows["Actual"], label = "Actual", color = "tab:blue", linewidth = 1)
    ax.plot(rows["tpep_pickup_datetime"], rows["Predicted"], label = "Predicted", color = "tab:orange", linewidth = 1, alpha = 0.8)
    ax.set_title(title)
    ax.set_xlabel("Pickup date and hour")
    ax.set_ylabel("Pickups per hour")
    ax.legend()
    ax.tick_params(axis = "x", rotation = 30)
fig.tight_layout()
plt.show()
